<a href="https://colab.research.google.com/github/pymood1/GPT-SoVITS-improve/blob/main/Colab-Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS Improve — Colab Inference\n\nOfficial Colab-Inference flow adapted for pymood1/GPT-SoVITS-improve.\n\n- syncs the fork main branch\n- keeps the official CU126/HF installer path\n- supports GPT, SoVITS and emotion-controller downloads\n- provides the regular WebUI and an optional local emotion API workflow\n\nThe WebUI uses a temporary Gradio share link in Colab.

## 1. Environment setup\n\nRun once per fresh Colab runtime.

In [ ]:
%%writefile /content/setup_gpt_sovits_improve.sh
set -euo pipefail

REPO_DIR="/content/GPT-SoVITS"
REPO_URL="https://github.com/pymood1/GPT-SoVITS-improve.git"

cd /content

if [ -d "${REPO_DIR}/.git" ]; then
    echo "[setup] existing checkout found; syncing main"
    cd "${REPO_DIR}"
    git fetch origin main
    git checkout main
    git reset --hard origin/main
else
    echo "[setup] cloning GPT-SoVITS-improve"
    git clone --branch main --single-branch "${REPO_URL}" "${REPO_DIR}"
    cd "${REPO_DIR}"
fi

git submodule update --init --recursive || true

mkdir -p GPT_weights
mkdir -p SoVITS_weights
mkdir -p emotion_controller_weights

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    echo "[setup] conda env GPTSoVITS already exists"
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS
python -m pip install -U pip
python -m pip install ipykernel

# Keep the official Colab installation path.
bash install.sh --device CU126 --source HF

echo
echo "[setup] repository HEAD:"
git rev-parse HEAD

In [ ]:
# Same Conda bootstrap flow as the official Colab notebook.
%pip install -q condacolab
import condacolab
condacolab.install_from_url(
    "https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh"
)

!cd /content && bash setup_gpt_sovits_improve.sh

## 2. Sync only\n\nUse this after installation when you only want the latest fork main.

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/GPT-SoVITS")
subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("GPT-SoVITS-improve HEAD:", head)

## 3. Optional Google Drive mount

In [ ]:
from google.colab import drive\ndrive.mount("/content/drive")

## 4. Optional Hugging Face custom model download\n\nFor private repositories, place HF_TOKEN in Colab Secrets. Empty filenames are skipped.

In [ ]:
from pathlib import Path
import shutil

REPO_DIR = Path("/content/GPT-SoVITS")

# ===== Modify these =====
HF_REPO_ID = ""
HF_REVISION = "main"
GPT_FILE = ""
SOVITS_FILE = ""
CONTROLLER_FILE = ""

if HF_REPO_ID:
    from huggingface_hub import hf_hub_download
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

    targets = [
        (GPT_FILE, REPO_DIR / "GPT_weights"),
        (SOVITS_FILE, REPO_DIR / "SoVITS_weights"),
        (CONTROLLER_FILE, REPO_DIR / "emotion_controller_weights"),
    ]
    for filename, out_dir in targets:
        if not filename:
            continue
        out_dir.mkdir(parents=True, exist_ok=True)
        cached = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=filename,
            revision=HF_REVISION,
            token=HF_TOKEN,
        )
        dst = out_dir / Path(filename).name
        shutil.copy2(cached, dst)
        print("downloaded:", dst)
else:
    print("HF_REPO_ID is empty — skipping custom model download.")

## 5. Verify installation and model folders

In [ ]:
from pathlib import Path
import subprocess
REPO_DIR = Path("/content/GPT-SoVITS")
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip())
for rel in [
    "GPT_SoVITS/pretrained_models/gsv-v5-pretrained",
    "GPT_weights",
    "SoVITS_weights",
    "emotion_controller_weights",
]:
    p = REPO_DIR / rel
    print(f"\n[{rel}]")
    if p.exists():
        for item in sorted(p.iterdir())[:30]:
            print(" ", item.name)
    else:
        print("  MISSING")

# 6A. Launch WebUI\n\nSame launch style as the official Colab-Inference notebook.

In [ ]:
!cd /content/GPT-SoVITS && \\\nsource activate GPTSoVITS && \\\nexport is_share=True && \\\npython webui.py

# 6B. Optional emotion-controller API mode\n\nUse this instead of the WebUI when testing the fork emotion controller. The API stays local on 127.0.0.1.

In [ ]:
import subprocess, socket, time
from pathlib import Path
REPO_DIR = Path("/content/GPT-SoVITS")
API_PORT = 9880
if "API_PROC" in globals() and API_PROC.poll() is None:
    API_PROC.terminate()
    try:
        API_PROC.wait(timeout=10)
    except subprocess.TimeoutExpired:
        API_PROC.kill()
API_PROC = subprocess.Popen(
    ["conda", "run", "--no-capture-output", "-n", "GPTSoVITS", "python", "api_v2.py", "-a", "127.0.0.1", "-p", str(API_PORT)],
    cwd=REPO_DIR,
)
deadline = time.time() + 120
while time.time() < deadline:
    if API_PROC.poll() is not None:
        raise RuntimeError(f"api_v2.py exited early with code {API_PROC.returncode}")
    with socket.socket() as s:
        s.settimeout(0.5)
        if s.connect_ex(("127.0.0.1", API_PORT)) == 0:
            print(f"API ready: http://127.0.0.1:{API_PORT}")
            break
    time.sleep(1)
else:
    raise RuntimeError("API startup timed out")

### Load optional GPT / SoVITS / emotion-controller weights\n\nThe emotion controller must match the semantic S2 identity used during controller training.

In [ ]:
import requests
from pathlib import Path
REPO_DIR = Path("/content/GPT-SoVITS")
BASE_URL = "http://127.0.0.1:9880"
GPT_WEIGHTS = ""
SOVITS_WEIGHTS = ""
CONTROLLER_WEIGHTS = ""
def call(endpoint, **params):
    r = requests.get(BASE_URL + endpoint, params=params, timeout=120)
    print(endpoint, r.status_code, r.text[:1000])
    r.raise_for_status()
    return r
if GPT_WEIGHTS:
    call("/set_gpt_weights", weights_path=GPT_WEIGHTS)
if SOVITS_WEIGHTS:
    call("/set_sovits_weights", weights_path=SOVITS_WEIGHTS)
if CONTROLLER_WEIGHTS:
    call("/set_emotion_controller", weights_path=CONTROLLER_WEIGHTS)

### Synthesize\n\nEMOTION_ID=None gives ordinary inference. Long-form planner work is not included yet; cut0-cut5 remain available.

In [ ]:
from pathlib import Path
import requests
from IPython.display import Audio, display
BASE_URL = "http://127.0.0.1:9880"
REF_AUDIO_PATH = ""
PROMPT_TEXT = ""
PROMPT_LANG = "ko"
TEXT = "안녕하세요. 이것은 GPT-SoVITS Improve의 Colab 추론 테스트입니다."
TEXT_LANG = "ko"
TEXT_SPLIT_METHOD = "cut0"
EMOTION_ID = None
EMOTION_STRENGTH = 1.0
if not REF_AUDIO_PATH:
    raise ValueError("REF_AUDIO_PATH를 지정하세요.")
payload = {
    "text": TEXT,
    "text_lang": TEXT_LANG,
    "ref_audio_path": REF_AUDIO_PATH,
    "prompt_text": PROMPT_TEXT,
    "prompt_lang": PROMPT_LANG,
    "text_split_method": TEXT_SPLIT_METHOD,
    "batch_size": 1,
    "parallel_infer": True,
    "use_cuda_graph": True,
    "media_type": "wav",
    "streaming_mode": 0,
    "emotion_id": EMOTION_ID,
    "emotion_strength": EMOTION_STRENGTH,
    "seed": -1,
}
r = requests.post(BASE_URL + "/tts", json=payload, timeout=300)
print("status:", r.status_code)
if r.status_code != 200:
    print(r.text[:3000])
    r.raise_for_status()
out_path = Path("/content/gpt_sovits_output.wav")
out_path.write_bytes(r.content)
print("saved:", out_path, out_path.stat().st_size, "bytes")
display(Audio(str(out_path)))

## 7. Stop local API

In [ ]:
if "API_PROC" in globals() and API_PROC.poll() is None:
    API_PROC.terminate()
    try:
        API_PROC.wait(timeout=10)
    except subprocess.TimeoutExpired:
        API_PROC.kill()
    print("API stopped")
else:
    print("API is not running")